In [ ]:
%pip install gymnasium[classic-control] tensorflow numpy matplotlib qiskit qiskit-aer pylatexenc tqdm IPython

In [2]:
import gymnasium
from gymnasium import logger as gymlogger
from gymnasium.wrappers import RecordVideo
import tensorflow as tf
import numpy as np
import random
import matplotlib
import matplotlib.pyplot as plt
import math
import copy
import glob
import io
import base64
from IPython.display import HTML
from IPython import display as ipythondisplay

from qiskit import QuantumCircuit
from qiskit_aer.primitives import SamplerV2 as Sampler
from tqdm import tqdm

In [3]:
"""
Utility functions to enable video recording of gym environment and displaying it
To enable video, just do "env = wrap_env(env)""
"""

def show_video():
  mp4list = glob.glob('./CartPole_videos/*.mp4')
  if len(mp4list) > 0:
    mp4 = mp4list[0]
    video = io.open(mp4, 'r+b').read()
    encoded = base64.b64encode(video)
    ipythondisplay.display(HTML(data='''<video alt="test" autoplay 
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
  else: 
    print("Could not find video")
    
def wrap_env(env):
  env = RecordVideo(env, video_folder="./CartPole_videos")
  return env

In [4]:
class NeuralNetwork:
  def __init__(self, env):
      self.env = env
      self.weightsIn = np.array([np.random.uniform(-1, 1, 2), np.random.uniform(-1, 1, 2), np.random.uniform(-1, 1, 2), np.random.uniform(-1, 1, 2)]) 
      self.score = 0

  def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

  def feed_forward(self, obs):
      self.output = self.sigmoid(np.dot(obs, self.weightsIn))
      action = np.argmax(self.output)
      return action

  def run3(self, truth):
      tempscore = 0
      self.run(truth)
      tempscore += self.score
      self.run(truth)
      tempscore += self.score
      self.run(truth)
      tempscore += self.score
      self.score = tempscore/3

  def run(self, truth):
        self.score = 0
        holder = self.env.reset()
        action = self.feed_forward(holder[0])
        new_obs, reward, is_done, truncation, _ = self.env.step(action)
        while not is_done and not truncation:
            self.score += reward
            if truth is True:
              self.env.render()
            action = self.feed_forward(np.array(new_obs))
            new_obs, reward, is_done, truncation, _ = self.env.step(action)
            #plt.imshow(env.render('rgb_array'))
            
  def fitness(self):
      return self.score

  def weights(self):
      return self.weightsIn #, self.weightsHid

  def setWeights(self, weightsIn): #, weightsHid):
      self.weightsIn = weightsIn
      #self.weightsHid = weightsHid

In [5]:
#Test one NN
env = gymnasium.make("CartPole-v1", render_mode="rgb_array")
env = wrap_env(env)
NN = NeuralNetwork(env)
NN.run3(True)

env.close()
show_video()

c:\Users\qkrtn\Desktop\QCS-Group-Project\venv\lib\site-packages\gymnasium\wrappers\rendering.py:283: UserWarning: WARN: Overwriting existing videos at c:\Users\qkrtn\Desktop\QCS-Group-Project\CartPole_videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [6]:
def generate_population (n):
  parents = []
  env = gymnasium.make("CartPole-v1", render_mode="rgb_array")
  for t in range (n):
    generated = NeuralNetwork(env)
    parents.append(generated)
  return parents

#parents = generate_population(64)

In [7]:
def fit_func(generated):
  generated.run(False)
  return generated.fitness()

In [8]:
env = gymnasium.make("CartPole-v1", render_mode="rgb_array")
generated = NeuralNetwork(env)
fit_func(generated)

37.0

In [9]:
n = 2
top = n/2
newpop = 1

In [10]:
# Initialize quantum circuit.
def initialize(num_qubits):
    qc = QuantumCircuit(num_qubits)
    for i in range(num_qubits):
        qc.h(i)

    return qc

# Measure quantum circuit.
def measure(qc):
    qc_measure = copy.deepcopy(qc)
    qc_measure.measure_all()

    simulator = Sampler()
    job = simulator.run([qc_measure], shots=1)
    result = job.result()
    counts = result[0].data.meas.get_counts()
    binary_string = list(counts.keys())[0]

    return [int(bit) for bit in binary_string]

# Map binary string to weights.
def binary_to_weights(binary_string, shape):
    # Map 0 -> -1 and 1 -> 1.
    weights = np.array(binary_string) * 2 - 1

    return weights.reshape(shape)

# Evaluate fitness.
def evaluate(env, weights):
    total_reward = 0.0
    obs, _ = env.reset()

    done = False
    while not done:
        action = np.argmax(np.dot(obs, weights))
        obs, reward, done, _, _ = env.step(action)
        total_reward += reward

    return total_reward

# Update quantum circuit.
def update(qc, thetas):
    for i, theta in enumerate(thetas):
        qc.ry(theta, i)
    
    return qc

# Calculate rotation angles.
def get_thetas(current_solution, best_solution, current_fitness, best_fitness, num_qubits):
    thetas = []
    for i in range(num_qubits):
        if current_solution[i] == best_solution[i]:
            thetas.append(0.01 * np.pi if current_fitness < best_fitness else 0.05 * np.pi)
        else:
            thetas.append(-0.01 * np.pi if current_fitness < best_fitness else -0.05 * np.pi)

    return thetas

# Execute Quantum Genetic Algorithm.
def gqa(env, num_qubits, max_generations, max_parameters):
    qc = initialize(num_qubits)
    best_solution = None
    best_fitness = -np.inf
    fitnesses = []

    generation, current_parameters = 0, 0
    while generation < max_generations:
        binary_solution = measure(qc)
        weights = binary_to_weights(binary_solution, (4, 2))
        fitness = evaluate(env, weights)

        if fitness > best_fitness:
            best_fitness = fitness
            best_solution = binary_solution

        fitnesses.append(fitness)
        average = np.average(fitnesses)

        print(f"Generation {generation + 1}: Best Fitness = {best_fitness:.3f} Average Fitness = {average:.3f}")

        if len(fitnesses) == 10:
            if average > 400.0:
                break
            else:
                fitnesses.pop(0)

        if current_parameters < max_parameters:
            thetas = get_thetas(binary_solution, best_solution, fitness, best_fitness, num_qubits)
            qc = update(qc, thetas)
            current_parameters += num_qubits

        generation += 1

    return best_solution, best_fitness, qc

In [11]:
# Visualization
def visualize(env, best_solution):
    weights = binary_to_weights(best_solution, (4, 2))
    obs, _ = env.reset()
    done = False
    total_reward = 0.0

    while not done:
        action = np.argmax(np.dot(obs, weights))
        obs, reward, done, _, _ = env.step(action)
        total_reward += reward
        env.render()
    print(f"Total Reward: {total_reward}")

    env.close()
    show_video()

In [12]:
# Main Execution

env = gymnasium.make("CartPole-v1", render_mode="rgb_array")
env = wrap_env(env)

num_qubits = 8
max_generations = 10_000
max_parameters = 32

best_solution, best_fitness, qc = gqa(env, num_qubits, max_generations, max_parameters)

print(f"Best Solution: {best_solution}")
print(f"Best Fitness: {best_fitness}")
print(qc)

visualize(env, best_solution)

c:\Users\qkrtn\Desktop\QCS-Group-Project\venv\lib\site-packages\gymnasium\wrappers\rendering.py:283: UserWarning: WARN: Overwriting existing videos at c:\Users\qkrtn\Desktop\QCS-Group-Project\CartPole_videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Generation 1: Best Fitness = 68.000 Average Fitness = 68.000
Generation 2: Best Fitness = 68.000 Average Fitness = 38.000
Generation 3: Best Fitness = 68.000 Average Fitness = 40.667
Generation 4: Best Fitness = 194.000 Average Fitness = 79.000
Generation 5: Best Fitness = 194.000 Average Fitness = 65.000
Generation 6: Best Fitness = 194.000 Average Fitness = 55.833
Generation 7: Best Fitness = 194.000 Average Fitness = 57.000
Generation 8: Best Fitness = 194.000 Average Fitness = 50.875
Generation 9: Best Fitness = 194.000 Average Fitness = 51.556
Generation 10: Best Fitness = 194.000 Average Fitness = 50.200
Generation 11: Best Fitness = 194.000 Average Fitness = 44.200
Generation 12: Best Fitness = 331.000 Average Fitness = 76.500
Generation 13: Best Fitness = 331.000 Average Fitness = 72.800
Generation 14: Best Fitness = 331.000 Average Fitness = 54.400
Generation 15: Best Fitness = 27720.000 Average Fitness = 2825.500
Best Solution: [0, 0, 1, 1, 0, 1, 0, 1]
Best Fitness: 27720.0
 